In [ ]:
# Uninstall the currently installed Transformers library
!pip uninstall -y transformers

# Install Transformers version 4.46.3
!pip install transformers==4.46.3

In [ ]:
# Import urlopen to read data directly from a URL
# it is used to open web pages and downloading files
from urllib.request import urlopen

# Import the Image class from the Pillow library
# Pillow (PIL = Python Imaging Library) is used for:Opening images,Resizing images,Cropping images,Rotating images
from PIL import Image

# URL of the AI-generated puppy image
puppy_path = "https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/chapter09/images/puppy.png"

# Download the image, open it, and convert it to RGB format (R=RED ,G=GREEN ,BLUE)
image = Image.open(urlopen(puppy_path)).convert("RGB")

# Define the correct caption (ground truth) for the image
caption = "a puppy playing in the snow"

In [ ]:
# Import the CLIP(Contrastive Language–Image Pretraining) tokenizer, processor, and model from Transformers
# It is a multimodal model developed by OpenAI.
# Together they allow CLIP to process text, process images, and generate embeddings
from transformers import CLIPTokenizerFast, CLIPProcessor, CLIPModel

# Hugging Face model identifier for OpenAI's CLIP model
model_id = "openai/clip-vit-base-patch32"

# Load the tokenizer to preprocess text inputs
clip_tokenizer = CLIPTokenizerFast.from_pretrained(model_id)

# Load the processor to preprocess images (and optionally text)
clip_processor = CLIPProcessor.from_pretrained(model_id)

# Load the pretrained CLIP model
model = CLIPModel.from_pretrained(model_id)

In [ ]:
# Convert the caption into token IDs and PyTorch tensors
inputs = clip_tokenizer(caption, return_tensors="pt")

# Display the tokenized output
inputs

In [ ]:
# Convert our input back to tokens
clip_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

In [ ]:
# Generate the text embedding from the tokenized caption
# ** is the dictionary unpacking operator.
# It takes every key-value pair in a dictionary and passes them as named (keyword) arguments to a function.
text_embedding = model.get_text_features(**inputs)

# Display the shape (dimensions) of the generated embedding
text_embedding.shape

In [ ]:
# Preprocess the image and convert it into a PyTorch tensor
processed_image = clip_processor(
    text=None,              # No text is being processed, only the image
    images=image,           # Input image
    return_tensors="pt"     # Return the output as PyTorch tensors
)["pixel_values"]

# Display the shape of the processed image tensor
processed_image.shape

In [ ]:
# Import the PyTorch library
import torch

# Import NumPy for numerical operations
import numpy as np

# Import Matplotlib for displaying images
import matplotlib.pyplot as plt

# Remove the batch dimension from the image tensor
img = processed_image.squeeze(0)

# Rearrange the dimensions from (Channels, Height, Width) to (Width, Height, Channels)
img = img.permute(*torch.arange(img.ndim - 1, -1, -1))

# Swap the first two axes to get (Height, Width, Channels)(i=Width,j=height ,k=Channels )
img = np.einsum("ijk->jik", img)

# Display the processed image
plt.imshow(img)

# Hide the x-axis and y-axis
plt.axis("off")

In [ ]:
# Generate an embedding (feature vector) for the processed image
image_embedding = model.get_image_features(processed_image)

# Display the shape (dimensions) of the image embedding
image_embedding.shape

In [ ]:
# Normalize the text embedding to have unit length
text_embedding /= text_embedding.norm(dim=-1, keepdim=True)

# Normalize the image embedding to have unit length
image_embedding /= image_embedding.norm(dim=-1, keepdim=True)

# Detach the text embedding from PyTorch's computation graph,
# move it to CPU, and convert it to a NumPy array
text_embedding = text_embedding.detach().cpu().numpy()

# Detach the image embedding from PyTorch's computation graph,
# move it to CPU, and convert it to a NumPy array
image_embedding = image_embedding.detach().cpu().numpy()

# Compute the similarity score between the text and image embeddings
score = np.dot(text_embedding, image_embedding.T)

# Display the similarity score
score

In [ ]:
# Import the processor used to prepare images and text for BLIP-2
from transformers import AutoProcessor, Blip2ForConditionalGeneration

# Import PyTorch
import torch

# Load the BLIP-2 processor
blip_processor = AutoProcessor.from_pretrained(
    "Salesforce/blip2-opt-2.7b"
)

# Load the BLIP-2 model with half-precision (FP16) to reduce GPU memory usage
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16
)

# Check if a CUDA GPU is available; otherwise use the CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# Move the model to the selected device (GPU or CPU)
model.to(device)

In [ ]:
# URL of the supercar image stored on GitHub
car_path = "https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/chapter09/images/car.png"

# Download the image, open it with Pillow, and convert it to RGB format
image = Image.open(urlopen(car_path)).convert("RGB")

# Display the loaded image
image

In [ ]:
# Preprocess the image and convert it into PyTorch tensors
inputs = blip_processor(
    image,
    return_tensors="pt"
).to(device, torch.float16)

# Display the shape of the processed image tensor
inputs["pixel_values"].shape

In [1]:
# Display the tokenizer used internally by the BLIP-2 processor
blip_processor.tokenizer

In [1]:
# Define the input text
text = "Her vocalization was remarkably melodic"

# Process both the image and the text together
token_ids = blip_processor(
    image,
    text=text,
    return_tensors="pt"
)

# Move the tensors to the selected device (GPU/CPU),
# convert them to float16 where applicable, and extract the input_ids
token_ids = token_ids.to(device, torch.float16)["input_ids"][0]

# Convert the token IDs back into readable tokens
tokens = blip_processor.tokenizer.convert_ids_to_tokens(token_ids)

# Display the tokens
tokens

In [1]:
# Replace the special space marker "Ġ" with "_" in every token
tokens = [token.replace("Ġ", "_") for token in tokens]

# Display the modified tokens
tokens

In [1]:
# Load the supercar image from the URL and convert it to RGB format
image = Image.open(urlopen(car_path)).convert("RGB")

# Preprocess the image, convert it into PyTorch tensors,
# move it to the selected device (CPU/GPU), and use float16 precision
inputs = blip_processor(
    image,
    return_tensors="pt"
).to(device, torch.float16)

# Display the loaded image
image

In [1]:
# Display all the keys present in the inputs dictionary
print(inputs.keys())

# Display the complete contents of the inputs dictionary
print(inputs)

In [2]:
# Preprocess the image and create an empty text prompt
inputs = blip_processor(
    image,
    text="",
    return_tensors="pt"
).to(device, torch.float16)

# Generate token IDs (caption) from the image
generated_ids = model.generate(
    **inputs,
    max_new_tokens=20
)

# Convert the generated token IDs into readable text
generated_text = blip_processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)

# Extract the first generated caption and remove extra spaces
generated_text = generated_text[0].strip()

# Display the generated caption
generated_text

In [2]:
# Create a prompt asking a question about the image
prompt = "Question: Write down what you see in this picture. Answer:"

# Preprocess both the image and the text prompt
inputs = blip_processor(
    image,
    text=prompt,
    return_tensors="pt"
).to(device, torch.float16)

# Generate an answer from the BLIP-2 model
generated_ids = model.generate(
    **inputs,
    max_new_tokens=30
)

# Convert the generated token IDs into readable text
generated_text = blip_processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)

# Extract the first answer and remove extra spaces
generated_text = generated_text[0].strip()

# Display the generated answer
generated_text

In [2]:
# Create a chat-style prompt with previous question-answer context
prompt = (
    "Question: Write down what you see in this picture. "
    "Answer: A sports car driving on the road at sunset. "
    "Question: What would it cost me to drive that car? "
    "Answer:"
)

# Preprocess both the image and the chat-style prompt
inputs = blip_processor(
    image,
    text=prompt,
    return_tensors="pt"
).to(device, torch.float16)

# Generate the model's response
generated_ids = model.generate(
    **inputs,
    max_new_tokens=30
)

# Convert the generated token IDs into readable text
generated_text = blip_processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)

# Extract the first response and remove extra spaces
generated_text = generated_text[0].strip()

# Display the generated answer
generated_text

In [2]:
# Import HTML display functions from Jupyter Notebook
# HTML → Displays formatted text (bold labels like USER and BLIP-2).
# display() → Shows widgets on the notebook.
from IPython.display import HTML, display

# Import interactive widgets
# ipywidgets → Creates interactive text boxes and output areas.
import ipywidgets as widgets


# Function that runs whenever the user enters text
def text_eventhandler(*args):

    # Get the text entered by the user
    question = args[0]["new"]

    # Continue only if the user typed something
    if question:

        # Clear the text box after pressing Enter
        args[0]["owner"].value = ""

        # Create the prompt
        if not memory:
            # First question (no previous conversation)
            prompt = "Question: " + question + " Answer:"
        else:
            # Format previous conversation
            template = "Question: {} Answer: {}."

            # Build conversation history
            prompt = " ".join(
                [
                    template.format(memory[i][0], memory[i][1])
                    for i in range(len(memory))
                ]
            ) + " Question: " + question + " Answer:"

        # Convert image and prompt into model inputs
        inputs = blip_processor(
            image,
            text=prompt,
            return_tensors="pt"
        )

        # Move inputs to GPU/CPU
        inputs = inputs.to(device, torch.float16)

        # Generate the answer
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=100
        )

        # Convert generated token IDs into text
        generated_text = blip_processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        # Remove any accidentally generated next question
        generated_text = generated_text[0].strip().split("Question")[0]

        # Save current question and answer into memory
        memory.append((question, generated_text))

        # Display the user's question
        output.append_display_data(
            HTML("<b>USER:</b> " + question)
        )

        # Display the model's answer
        output.append_display_data(
            HTML("<b>BLIP-2:</b> " + generated_text)
        )

        # Add a blank line
        output.append_display_data(
            HTML("<br>")
        )


# Create a text input widget
in_text = widgets.Text()

# Only trigger after pressing Enter
in_text.continuous_update = False

# Connect the textbox with the event handler
in_text.observe(text_eventhandler, "value")

# Create an output area for conversation
output = widgets.Output()

# Initialize conversation memory
memory = []

# Display the chatbot interface
display(
    widgets.VBox(
        children=[output, in_text],
        layout=widgets.Layout(
            display="inline-flex",
            flex_flow="column-reverse"
        ),
    )
)